# roberta2roberta Approach

This method uses the fine-tuned RoBERTa model to tag temporal text. Regex is used to filter out tagged text that is not for clock-related times.

In [ ]:
%pip install transformers tqdm torch nltk -q

## Imports & Config

In [ ]:
import re
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# bypass SSL issue
# see: https://github.com/gunthercox/ChatterBot/issues/930#issuecomment-322111087
import nltk
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

# Match any opening tag with a clock-time value (T HH:MM) (e.g., value="1998-12-23T16:00"),
# and capture text until the next tag. We don't match the closing tags since the model
# generates them inconsistently.
CLOCK_TIME_PATTERN = re.compile(
    r'<[^>]*[Vv]alue="[^"]*T(\d{2}:\d{2})[^"]*"[^>]*>\s*(.*?)\s*<',
    re.IGNORECASE,
)

# Use the same approach for all TIMEX3 tags: capture the type, value, and text span.
# This is so we can see what else roberta2roberta tags in the sentences that are tagged
# for clock times.
ALL_TIMEX3_PATTERN = re.compile(
    r'<[^>]*[Vv]alue="([^"]+)"[^>]*>\s*(.*?)\s*<',
    re.IGNORECASE,
)

# Config for the model
BOOK_FILE = "gatsby.txt"
OUTPUT_FILE = "roberta_tagged.json"
CONTEXT_WINDOW = 1  # number of sentences to process at a time

## Extraction: Tag temporal sentences and filter out non clock-based text.

roberta2roberta outputs the original sentence with inline TIMEX3 XML tags. For example:

```xml
Some text before... <TIMEX3 type="TIME" value="T00:00">the stroke of midnight</TIMEX3> some text after.
Other text before... <TIMEX3 type="TIME" value="T15:30">half past three</TIMEX3> other text after.
```

The `value` attribute is ISO 8601 complient, so specific clock times normalize to `THH:MM`.

In [ ]:
def extract_clock_quotes(filepath, context_window=1, max_sentences=None):
    """Return sentences where the model produces a clock-time TIMEX3 value (THH:MM).

    Set max_sentences to a small number (e.g. 100) to test on a subset first.
    """
    print("Loading RoBERTa2RoBERTa Temporal Tagger...")
    model_name = "satyaalmasian/temporal_tagger_roberta2roberta"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    print(f"Reading: {filepath}")
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    sentences = sent_tokenize(text.replace("\n", " "))
    sentences = [s.strip() for s in sentences if s.strip()]

    if max_sentences is not None:
        sentences = sentences[:max_sentences]
        print(f"Processing first {max_sentences} sentences (test mode).")

    clock_quotes = []
    print("\nTagging sentences...")
    for i, sentence in enumerate(tqdm(sentences, desc="RoBERTa2RoBERTa tagging")):
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
        output_ids = model.generate(**inputs, max_new_tokens=128)
        annotated = tokenizer.decode(output_ids[0], skip_special_tokens=True)

        clock_matches = CLOCK_TIME_PATTERN.findall(annotated)
        if clock_matches:
            start = max(0, i - context_window)
            end = min(len(sentences), i + context_window + 1)
            all_entities = [
                {"value": v, "text": span}
                for v, span in ALL_TIMEX3_PATTERN.findall(annotated)
            ]
            clock_quotes.append({
                "sentence_index": i,
                "target_sentence": sentence,
                "annotated_sentence": annotated,
                "full_context": " ".join(sentences[start:end]),
                "clock_entities": [{"clock_time": t, "text": span} for t, span in clock_matches],
                "all_timex3_entities": all_entities,
            })

    print(f"\nFound {len(clock_quotes)} clock-based quotes.")
    return clock_quotes

In [ ]:
'''
Extract the tagged text and 
write the output to a JSON file.
'''
clock_quotes = extract_clock_quotes(BOOK_FILE, context_window=CONTEXT_WINDOW)


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(clock_quotes, f, indent=4, ensure_ascii=False)
print(f"Saved {len(clock_quotes)} clock quotes to {OUTPUT_FILE}")